In [ ]:
import json
from pathlib import Path
from collections import defaultdict

%load_ext autoreload
%autoreload 2

from utgen.raggraph.walker import build_graph_from_directory
from utgen.raggraph.utils import get_node_context
from utgen.test_generation_crew.crew import TestGenerationCrew
from utgen.validation import validate_individual_test, save_and_clean_tests

In [ ]:
g = build_graph_from_directory(code_path="../src", save_graph=False)

In [ ]:
# Define defaultdict of dicts
tests_results = defaultdict(dict)

# TODO: afegir guardrails que falten
test_generator = TestGenerationCrew(guardrail_max_retries=3, verbose=False)

# Per a cada node del graf, obtenim el seu context i creem el test
for node_id, data in list(g.nodes(data=True)):
    if data["type"] in ["function"]:  # ["function", "method"]
        print(f"Generating tests for node: {node_id}")
        # Get context
        context = get_node_context(g=g, node_id=node_id)

        # Generate tests
        inputs = {"graph_context": context}
        response = test_generator.crew().kickoff(inputs=inputs)

        # Convert string to dictionary
        response_dict = json.loads(response.raw)

        # Store results
        p = Path(data["file"])
        new_filename = f"test_{p.stem}{p.suffix}"
        save_path = (p.parent / new_filename).as_posix()
        tests_results[save_path][node_id] = response_dict['tests']

In [ ]:
for save_path, nodes in tests_results.items():
    tests_que_han_passat = []
    for node_id, tests in nodes.items():
        # Validem el test generat
        base_import = 'from ' + node_id.split("::")[0][:-3].replace("/", ".") + ' import ' + node_id.split("::")[-1].split(".")[0]
        for name, values in tests.items():
            imports, code = values['imports'], values['code']
            # Afegim el import base per si no està present
            if base_import not in imports:
                print(f"⚠️ Import base `{base_import}` no present en el test `{name}`, afegint-lo...")
                imports.append(base_import)
            imports = "\n".join(imports)

            if validate_individual_test(import_code=imports, test_code=code):
                print(f"✅ Test `{name}` acceptat")
                tests_que_han_passat.append((imports, code))
            else:
                print(f"❌ Test `{name}` rebutjat")

    # Guardem i deixem el fitxer perfecte
    print(f"\nGuardant tests per `{save_path}`...")
    save_and_clean_tests(valid_tests=tests_que_han_passat, destination=f"../tests/{save_path}")

# TODO: run pytest coverage